# Pipeline
0. Get the list of required swc files
1. Load labels parquet
2. Import the swc file
3. simplify swc file
4. attach synapse labels + neuron type
5. save simplified file
6. convert to json for find-clumpiness
7. save json file
8. calculate clumpiness for each internal node
9. attach results to the labeled swc file
10. save results.

---
# Preprocessing step
1. Create metadata labels for each swc file
2. Look for the releveant swc files only (with the wanted type)
3. Unify them via the already created function in feather file ->>> Improvement

In [1]:
import os
import json
import numpy as np
import pandas as pd
import polars as pl
from tqdm import tqdm
from scripts.helpers import mkdir
from joblib import Parallel, delayed
from scripts.preprocessing import simplify_swc_topology, swc2json, get_neurons_info
from scripts.processing import generate_internal_subtrees

if False:
    # Type data located in the 
    path_swc_labels = os.path.join("data", "input_labels", "neuron_data_full_article_princeton.ftr")
    swc_labels = pd.read_feather(path_swc_labels)

    required_labels = ["super_class", ["central", "optic", "visual_centrifugal", "visual_projection"]]
    swc_labels = swc_labels.loc[swc_labels[required_labels[0]].isin(required_labels[1])]

-----
# Single-file hard coded pipeline example

In [2]:
if False:
    nueron_itr = 720575940609102805


    ####################################################################################################
    #  1. Load labels parquet
    # Parquet labels path
    prquet_labels_path = os.path.join("data", "input_labels", "swc_labels.parquet")

    # Load exactly the labels of the example swc file
    parquet_labels = pl.scan_parquet(prquet_labels_path)

    # Only the relevnt column in the parquet file
    labels_parquet = parquet_labels.select(["neuron", "node_id", "type"]).filter(pl.col("neuron") == str(nueron_itr)).collect().to_pandas()


    ####################################################################################################
    #  2. Import the swc file
    neuron_path = os.path.join("data","input_swc", "sk_lod1_783_healed", f"{nueron_itr}.swc")
    neuron_swc = pd.read_csv(neuron_path, 
                            comment='#', 
                            header=None, 
                            sep=r'\s+', 
                            names=["node_id", "swc_type", "x", "y", "z", "r", "parent"])


    ####################################################################################################
    #  3. simplify swc file
    simple_swc = simplify_swc_topology(neuron_swc, swc_name=f"{nueron_itr}", save_csv=False)


    ####################################################################################################
    #  4. attach synapse labels + neuron type
    swc_labeled = pd.merge(left=simple_swc, 
                        right=labels_parquet[["node_id", "type"]].drop_duplicates(), 
                        left_on="node_id", 
                        right_on="node_id", 
                        how="left")


    ####################################################################################################
    #  5. save simplified file
    for i in ["data", os.path.join("data", "input_swc"), os.path.join("data", "input_swc", "simplified")]:
        if os.path.exists(i) is False:
            os.mkdir(i)

    save_path = os.path.join("data", "input_swc", "simplified", f"{nueron_itr}.csv")
    swc_labeled.to_csv(save_path)


    ####################################################################################################
    #  6. convert to json for find-clumpiness
    swc2json(swc_dataset=swc_labeled,
            neuron_id=nueron_itr,
            save_json=True,
            save_path=os.path.join("data", "output_json"))


----
# Automated script pipeline


In [ ]:
def process_neuron(neuron_itr):
    # Force Polars to use a single thread to prevent nested parallelism crashes
    os.environ["POLARS_MAX_THREADS"] = "4"

    try:
        #########################
        #  1. Load labels parquet
        # Parquet labels path
        prquet_labels_path = os.path.join("data", "input_labels", "swc_labels.parquet")


        # Load exactly the labels of the example swc file
        parquet_labels = pl.scan_parquet(prquet_labels_path)


        # Only the relevnt column in the parquet file
        labels_parquet = parquet_labels.select(["neuron", "node_id", "type"]).filter(pl.col("neuron") == str(neuron_itr)).collect().to_pandas()


        #########################
        #  2. Import the swc file
        neuron_path = os.path.join("data","input_swc", "sk_lod1_783_healed", f"{neuron_itr}.swc")
        neuron_swc = pd.read_csv(neuron_path, 
                                 comment='#', 
                                 header=None, 
                                 sep=r'\s+', 
                                 names=["node_id", "swc_type", "x", "y", "z", "r", "parent"])


        #######################
        #  3. simplify swc file
        simple_swc = simplify_swc_topology(neuron_swc, swc_name=f"{neuron_itr}", save_csv=False)  


        #########################################
        #  4. attach synapse labels + neuron type
        swc_labeled = pd.merge(left=simple_swc, 
                               right=labels_parquet[["node_id", "type"]].drop_duplicates(), 
                               left_on="node_id", 
                               right_on="node_id", 
                               how="left")   


        ##########################
        #  5. save simplified file
        for i in ["data", os.path.join("data", "input_swc"), os.path.join("data", "input_swc", "simplified")]:
            if os.path.exists(i) is False:
                os.mkdir(i)

        save_path = os.path.join("data", "input_swc", "simplified", f"{neuron_itr}.csv")
        swc_labeled["type"] = swc_labeled.groupby("node_id")["type"].unique().apply(lambda X : X[0] if len(X) <= 1 else ",".join(X)) # joining labels if more then 2 per node
        swc_labeled = swc_labeled.drop_duplicates(subset=["node_id", "parent"], keep="first")                                        # dropping rows with the same parent+node_id
        swc_labeled.to_csv(save_path)


        #########################################
        #  6. convert to json for find-clumpiness
        swc2json(swc_dataset=swc_labeled.drop_duplicates(),
                 neuron_id=neuron_itr,
                 save_json=True,
                 save_path=os.path.join("data", "output_json"))


        ##################################################################
        # 7. Devide main tree to multiple sub-trees for each internal node
        # Example Usage:
        generate_internal_subtrees(input_json_path = os.path.join("data","output_json",f"{neuron_itr}_0.json"), 
                                   neuron_number = neuron_itr, 
                                   output_dir = os.path.join("data", "output_json"))
        

        return f"Success: {neuron_itr}"

    except Exception as e:
        return f"Error on {neuron_itr}: {e}"


if __name__ == '__main__':
    # Define paths
    swc_path = os.path.join("data", "input_swc", "sk_lod1_783_healed")
    labels_path = os.path.join("data", "input_labels", "processed_swc_data_princeton")
    prquet_labels_path = os.path.join("data", "input_labels", "swc_labels.parquet")

    # Safe folder creation before multiprocessing starts to avoid race conditions
    folders_to_create = ["data", 
                         os.path.join("data", "input_swc"), 
                         os.path.join("data", "input_swc", "simplified"),
                         os.path.join("data", "output_json")]
    
    for folder in folders_to_create:
        os.makedirs(folder, exist_ok=True)

    # Getting a list of the aviable SWC file in the swc input folder
    swc_files = [i.split(".")[0] for i in os.listdir(swc_path)]

    # Creating parquete file -> only relevent swc file by super-type
    get_neurons_info(overwrite_parquet=False)

    # Getting relevent swc
    parquet_labels = pl.scan_parquet(prquet_labels_path)
    labels_parquet = parquet_labels.select(["neuron"]).collect().to_pandas()

    # Getting list of relevent + real SWC file
    swc_relv = np.intersect1d(swc_files, labels_parquet.neuron.values)
    
    # Select the batch you want to run
    tasks = swc_relv[:20] 
    
    print(f"Starting processing of {len(tasks)} neurons...")
    
    # Execute in parallel using Joblib
    # n_jobs=4 limits the pool to 4 cores to prevent memory exhaustion. 
    # You can increase this if your system has plenty of RAM.
    results = Parallel(n_jobs=4, backend="loky")(delayed(process_neuron)(neuron) for neuron in tqdm(tasks))
    
    # Print any errors that were caught during execution
    for res in results:
        if "Error" in res:
            print(res)


> Function execution halted, old `swc_labels.parquet` file preserved.
Starting processing of 20 neurons...


100%|██████████| 20/20 [00:16<00:00,  1.22it/s]


In [ ]:
import os
import sys
import csv
import uuid
import shutil
import subprocess
from pathlib import Path
from joblib import Parallel, delayed
from tqdm import tqdm  # Imported for the progress bar

def _process_single_neuron(filepath):
    """
    Worker function executed in parallel. 
    Prints removed to prevent terminal output corruption.
    """
    neuron_id = filepath.stem
    temp_filename = f"temp_clump_{uuid.uuid4().hex}.csv"
    
    try:
        result = subprocess.run(
            ["find-clumpiness", "-e", "AllExclusive", "-i", str(filepath), "-f", "JSON"],
            capture_output=True,
            text=True,
            check=True
        )
        
        lines = result.stdout.strip().splitlines()
        
        if len(lines) <= 1:
            return None
            
        with open(temp_filename, 'w', newline='') as temp_out:
            for line in lines[1:]:
                parts = line.split(',')
                if len(parts) >= 3:
                    col1 = parts[0].strip()
                    col2 = parts[1].strip()
                    col3 = parts[2].strip()
                    
                    formatted_result = f'"{neuron_id}",{col1}-{col2},{col3}\n'
                    temp_out.write(formatted_result)
                    
        return temp_filename
        
    except subprocess.CalledProcessError:
        # Silently fail or log to a file instead of printing
        return None
    except FileNotFoundError:
        # Command not found
        return None

def call_clumpiness(INPUT_DIR=os.path.join("data", "output_json"),
                    OUTPUT_DIR=os.path.join("data", "output_clumpiness"),
                    OUTPUT_FILE=os.path.join("data", "clumpiness.csv"),
                    n_jobs=-1):
    
    Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

    processed_ids = set()
    output_path = Path(OUTPUT_FILE)
    
    if output_path.exists():
        with open(OUTPUT_FILE, 'r', newline='') as f:
            reader = csv.reader(f)
            next(reader, None) 
            for row in reader:
                if row:
                    clean_id = row[0].strip('"') 
                    processed_ids.add(clean_id)
    else:
        with open(OUTPUT_FILE, 'w', newline='') as f:
            f.write("neuron_id,groups,clumpiness\n")

    input_path = Path(INPUT_DIR)
    
    if not input_path.exists() or not input_path.is_dir():
        sys.exit(1)

    json_files = list(input_path.glob("*.json"))
    total_files = len(json_files)

    if total_files == 0:
        sys.exit(0)

    files_to_process = [f for f in json_files if f.stem not in processed_ids]
    total_to_process = len(files_to_process)

    if total_to_process <= 0:
        sys.exit(0)

    # --- EXECUTION WITH PROGRESS BAR ---
    
    # joblib returns a generator so tqdm can update the bar as each job completes
    parallel_jobs = Parallel(n_jobs=n_jobs, return_as="generator")(
        delayed(_process_single_neuron)(filepath) for filepath in files_to_process
    )
    
    # Wrap the generator in tqdm to render the progress bar
    temp_filepaths = list(tqdm(
        parallel_jobs, 
        total=total_to_process, 
        desc="Processing Neurons", 
        unit="file"
    ))

    # --- GATHER STEP ---
    
    with open(OUTPUT_FILE, 'a', newline='') as master_out:
        for temp_file in temp_filepaths:
            if temp_file is None or not os.path.exists(temp_file):
                continue
                
            with open(temp_file, 'r') as temp_in:
                shutil.copyfileobj(temp_in, master_out)
                
            os.remove(temp_file)

In [7]:
from scripts.helpers import read_json
from pathlib import Path

config = read_json()

In [30]:
if __name__ == '__main__':
    print("###########################################",
          "\n",
          "> Loading paths and creating folders. (1/5)")
    # Loading config presets
    config = read_json(path="config.json")
    swc_path, labels_path, prquet_labels_path, overwrite_info, data_limit, n_threads, n_jobs =  [config["swc_path"],
                                                                                                 config["labels_path"],
                                                                                                 config["prquet_labels_path"],
                                                                                                 config["overwrite_info"],
                                                                                                 config["data_limit"],
                                                                                                 config["n_threads"],
                                                                                                 config["n_jobs"]]

    # Force Polars to use a single thread to prevent nested parallelism crashes
    os.environ["POLARS_MAX_THREADS"] = str(n_threads)

    # Safe folder creation before multiprocessing starts to avoid race conditions
    folders_to_create = ["data", 
                         os.path.join("data", "input_swc"), 
                         os.path.join("data", "input_swc", "simplified"),
                         os.path.join("data", "output_json")]
    
    for folder in folders_to_create:
        os.makedirs(folder, exist_ok=True)


    swc_path = os.path.join(*swc_path.split(","))
    labels_path = os.path.join(*labels_path.split(","))
    prquet_labels_path = os.path.join(*prquet_labels_path.split(","))

    print("#################################################################",
          "\n",
          "> Creating unified labels file for every relevent SWC tree. (2/5)")
    # Getting a list of the aviable SWC file in the swc input folder
    swc_files = [i.split(".")[0] for i in os.listdir(swc_path)]

########################################### 
 > Loading paths and creating folders. (1/5)
################################################################# 
 > Creating unified labels file for every relevent SWC tree. (2/5)


In [ ]:
swc_path, labels_path, prquet_labels_path, overwrite_info, data_limit, n_threads, n_jobs =  read_json(path="config.json")


In [38]:
bool_val = "true"
overwrite_par = (True if bool_val == "true" else False)
overwrite_par

True

----

In [ ]:
import pandas as pd
import os
files_json = pd.read_csv(os.path.join("_misc", "json_files.csv"), header=None)
files_json.columns = ["full"]
files_json["short"] = files_json["full"].apply(lambda X: X.split("_")[0])

In [17]:
len(files_json["short"].unique())

116266

---

In [21]:
import os
import pandas as pd
from pathlib import Path
import pyarrow as pa
import pyarrow.parquet as pq
import itertools
from tqdm import tqdm
from joblib import Parallel, delayed

def process_clumpiness_csv(filepath_str: str):
    """
    Reads a single CSV, safely skips empty files, and appends ID columns.
    """
    filepath = Path(filepath_str)
    
    # Fast-skip: Files <= 32 bytes physically cannot contain data rows
    if os.path.getsize(filepath) <= 32:
        return None
        
    try:
        df = pd.read_csv(filepath)
        if df.empty:
            return None
            
        neuron_id, node_id = filepath.stem.split('_')
        
        df['neuron_id'] = neuron_id
        df['node_id'] = node_id
        
        return df
        
    except Exception:
        return None

def chunked_iterable(iterable, size):
    """Yields batches of a specified size from an iterable."""
    it = iter(iterable)
    while True:
        chunk = tuple(itertools.islice(it, size))
        if not chunk:
            break
        yield chunk

def compile_unified_dataset(input_directory: str, output_filepath: str, batch_size: int = 2000) -> None:
    """
    Iterates over CSVs and processes them using joblib for robust parallel execution.
    """
    def get_csv_files():
        with os.scandir(input_directory) as entries:
            for entry in entries:
                if entry.name.endswith('.csv') and entry.is_file():
                    yield entry.path

    writer = None
    n_jobs = min(4, (os.cpu_count() or 1))
    
    with tqdm(desc="Compiling Parquet", unit=" files") as pbar:
        for file_chunk in chunked_iterable(get_csv_files(), batch_size):
            
            # joblib handles the worker pool much more safely on Windows
            results = Parallel(n_jobs=n_jobs, backend="loky")(
                delayed(process_clumpiness_csv)(f) for f in file_chunk
            )
            
            valid_dfs = [df for df in results if df is not None]
            
            if valid_dfs:
                batch_df = pd.concat(valid_dfs, ignore_index=True)
                table = pa.Table.from_pandas(batch_df)
                
                if writer is None:
                    writer = pq.ParquetWriter(output_filepath, table.schema, compression='ZSTD')
                
                writer.write_table(table)
            
            pbar.update(len(file_chunk))
            
    if writer:
        writer.close()
    print("\nDataset compilation complete.")

if __name__ == "__main__":
    input_dir = os.path.join("data", "output_clumpiness")
    output_file = os.path.join("data", "unified_clumpiness.parquet")
    
    Path(output_file).parent.mkdir(parents=True, exist_ok=True)
    
    compile_unified_dataset(input_dir, output_file)

Compiling Parquet: 100000 files [01:08, 1452.51 files/s]


Dataset compilation complete.


In [ ]:
import os
import pandas as pd

df = pd.read_parquet(os.path.join("data", "unified_clumpiness.parquet"))

In [4]:
df

,property1,property2,value,neuron_id,node_id
0,post,post,-1.008826,720575940597944841,1773
1,post,pre,0.000000,720575940597944841,1773
2,pre,post,0.000000,720575940597944841,1773
3,pre,pre,1.000000,720575940597944841,1773
4,pre,pre,1.000000,720575940599457990,134
...,...,...,...,...,...
62970,post,post,1.000000,720575940661318017,2617
62971,post,pre,0.000000,720575940661318017,2617
62972,pre,post,0.000000,720575940661318017,2617
62973,pre,pre,1.000000,720575940661318017,2617
